In [1]:
import numpy as np
import pandas as pd
import time
import faiss
import hnswlib

from annoy import AnnoyIndex

In [2]:
def recall_at_k(I_pred, I_gt, k):
    correct = 0

    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))

    return correct / (len(I_pred) * k)

In [3]:
def run_experiment(n_data, dim, nq=100, k=10, seed=42):
    rng = np.random.RandomState(seed)

    X = rng.random((n_data, dim)).astype('float32')
    Q = rng.random((nq, dim)).astype('float32')

    rows = []

    flat = faiss.IndexFlatL2(dim)
    flat.add(X)

    start = time.time()
    _, gt = flat.search(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'Exact (FAISS Flat)', 'Build (s)': 0.0, 'Query (ms/query)': t_query, f'Recall@{k}': 1.0})

    ann = AnnoyIndex(dim, 'euclidean')

    start = time.time()
    for i in range(n_data):
        ann.add_item(i, X[i])
    ann.build(10)
    t_build = time.time() - start

    start = time.time()
    I = [ann.get_nns_by_vector(q, k) for q in Q]
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'ANNOY', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    nlist = max(1, int(np.sqrt(n_data)))

    quantizer = faiss.IndexFlatL2(dim)
    ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)

    start = time.time()
    ivf.train(X)
    ivf.add(X)
    t_build = time.time() - start

    ivf.nprobe = 10

    start = time.time()
    _, I = ivf.search(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'FAISS', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    hnsw = hnswlib.Index(space='l2', dim=dim)

    start = time.time()
    hnsw.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw.add_items(X)
    t_build = time.time() - start

    hnsw.set_ef(50)

    start = time.time()
    I, _ = hnsw.knn_query(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'HNSW', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    df = pd.DataFrame(rows)
    df.insert(0, 'Data/Dimensions', f'{n_data:,}/{dim}D')

    return df

In [4]:
n_big = 1_000_000

configs = [(1000, 2), (1000, 5), (n_big, 2), (n_big, 5)]

frames = []

for n_data, dim in configs:
    print(f'Running {n_data:,} data points, {dim}D ...')
    frames.append(run_experiment(n_data, dim))

results = pd.concat(frames, ignore_index=True)

results.round(4)

Running 1,000 data points, 2D ...
Running 1,000 data points, 5D ...
Running 1,000,000 data points, 2D ...
Running 1,000,000 data points, 5D ...


,Data/Dimensions,Algorithm,Build (s),Query (ms/query),Recall@10
0,"1,000/2D",Exact (FAISS Flat),0.0000,0.0100,1.000
1,"1,000/2D",ANNOY,0.0127,0.0252,0.993
2,"1,000/2D",FAISS,0.0028,0.0101,1.000
3,"1,000/2D",HNSW,0.0156,0.0201,1.000
4,"1,000/5D",Exact (FAISS Flat),0.0000,0.0000,1.000
5,"1,000/5D",ANNOY,0.0080,0.0254,0.962
6,"1,000/5D",FAISS,0.0000,0.0000,0.997
7,"1,000/5D",HNSW,0.0164,0.0100,1.000
8,"1,000,000/2D",Exact (FAISS Flat),0.0000,1.0533,1.000
9,"1,000,000/2D",ANNOY,8.0671,0.0939,0.998


In [5]:
query_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Query (ms/query)')

query_table = query_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

query_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.0252,0.0100,0.0101,0.0201
"1,000/5D",0.0254,0.0000,0.0000,0.0100
"1,000,000/2D",0.0939,1.0533,0.0251,0.0273
"1,000,000/5D",0.0000,1.2732,0.0277,0.0279


In [6]:
build_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Build (s)')

build_table = build_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

build_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.0127,0.0,0.0028,0.0156
"1,000/5D",0.0080,0.0,0.0000,0.0164
"1,000,000/2D",8.0671,0.0,7.6118,19.7331
"1,000,000/5D",7.2004,0.0,8.1307,32.7196


In [7]:
recall_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Recall@10')

recall_table = recall_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

recall_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.993,1.0,1.000,1.0
"1,000/5D",0.962,1.0,0.997,1.0
"1,000,000/2D",0.998,1.0,1.000,1.0
"1,000,000/5D",0.941,1.0,1.000,1.0


### Analysis – Lab Assignment 1

**Answer:** With 1,000 points, exact search is already fast (about 0.01 ms per query), so approximate search gives no benefit. With 1,000,000 points, exact search takes 1.05 to 1.27 ms, while FAISS and HNSW take about 0.027 ms (roughly 40 to 50 times faster). In 5D the recall of ANNOY drops (0.998 to 0.941 at 1,000,000 points), while FAISS and HNSW stay at about 1.0. HNSW is the most accurate but the slowest to build (20 to 33 s), FAISS is almost as fast with a build time of 7 to 8 s, and ANNOY has the lowest recall at its default settings.

*Note: the 1,000-point query times are below the timer resolution of `time.time()` (about 1 ms) and should not be compared.*